# IMPORT

In [1]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# DEFINE FOLDERS

In [2]:
STATION_CLUSTER_FOLDER = Path("station_clusters")
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
MODEL_OUTPUT_FOLDER = Path("cluster_models")

folders = [STATION_CLUSTER_FOLDER, DATASET_INPUT_FOLDER, MODEL_OUTPUT_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA AND CLUSTERS

In [4]:

df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/final_clean_data.csv")
clusters = pd.read_csv(f"{STATION_CLUSTER_FOLDER}/station_clusters_2_42.csv")

# Merge cluster info
df = df.merge(clusters, on="station_code", how="inner")

print("Rows:", df.shape)
print("Clusters:", df["cluster"].value_counts())
print(df.columns)


Rows: (6803688, 18)
Clusters: cluster
0    3458726
1    3344962
Name: count, dtype: int64
Index(['timestamp', 'station_code', 'latitude', 'longitude', 'capacity',
       'total_bikes_now', 'total_bikes_taken', 'temperature', 'wind_speed',
       'wind_direction', 'weather_code', 'is_day', 'hour', 'day_of_week',
       'is_weekend', 'sin_hour', 'cos_hour', 'cluster'],
      dtype='object')


# FEATURE SELECTION

In [5]:
FEATURES = [
    # Time
    "sin_hour",
    "cos_hour",
    "is_weekend",

    # Weather
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",

    # Station context
    "capacity",
]

TARGET = "total_bikes_now"


# TIME BASED 70-15-15 SPLIT

In [19]:
def time_split(df, train_frac=0.7, val_frac=0.15):
    df = df.sort_values("timestamp")

    n = len(df)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))

    train = df.iloc[:train_end]
    val   = df.iloc[train_end:val_end]
    test  = df.iloc[val_end:]

    return train, val, test


# TRAIN MODELS PER CLUSTER

In [ ]:
results = []

for cluster_id in sorted(df["cluster"].unique()):
    print(f"\n - Training cluster {cluster_id}")

    # Select cluster and sort by time
    df_c = df[df["cluster"] == cluster_id].sort_values("timestamp")

    # Create lag features
    df_c["total_bikes_lag1"] = df_c.groupby("station_code")["total_bikes_now"].shift(1)
    df_c["total_bikes_lag2"] = df_c.groupby("station_code")["total_bikes_now"].shift(2)
    df_c = df_c.dropna(subset=["total_bikes_lag1", "total_bikes_lag2"])

    # Update features
    FEATURES_LAG = FEATURES + ["total_bikes_lag1", "total_bikes_lag2"]

    # Time-based split
    train, val, test = time_split(df_c, train_frac=0.7, val_frac=0.15)
    X_train, y_train = train[FEATURES_LAG], train[TARGET]
    X_val, y_val     = val[FEATURES_LAG], val[TARGET]
    X_test, y_test   = test[FEATURES_LAG], test[TARGET]

    models = {
        "LinearRegression": LinearRegression(),
        "SGDRegressor": Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ]),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        ),
    }

    # Train and evaluate
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)

        results.append({
            "cluster": cluster_id,
            "model": name,
            "r2": r2,
            "mae": mae,
            "n_samples": len(df_c)
        })

        print(f"  {name:16s} → R^2={r2:.3f} | MAE={mae:.2f}")



 - Training cluster 0
  LinearRegression → R^2=0.954 | MAE=0.92
  SGDRegressor     → R^2=0.954 | MAE=0.91
  RandomForest     → R^2=0.957 | MAE=0.84

 - Training cluster 1
  LinearRegression → R^2=0.975 | MAE=1.09
  SGDRegressor     → R^2=0.975 | MAE=1.10
  RandomForest     → R^2=0.977 | MAE=1.01


# COMPARE RESULTS

In [21]:
results_df = pd.DataFrame(results)
results_df.sort_values(["cluster", "r2"], ascending=[True, False])

,cluster,model,r2,mae,n_samples
2,0,RandomForest,0.957164,0.840286,3457266
1,0,SGDRegressor,0.954185,0.909167,3457266
0,0,LinearRegression,0.954157,0.917324,3457266
5,1,RandomForest,0.976724,1.013360,3343550
3,1,LinearRegression,0.975355,1.094892,3343550
4,1,SGDRegressor,0.975321,1.102640,3343550


# GET BEST MODEL PER CLUSTER

In [22]:
best_models = {}

for cluster_id in results_df["cluster"].unique():
    best = results_df[results_df["cluster"] == cluster_id].sort_values(
        "r2", ascending=False
    ).iloc[0]

    best_models[cluster_id] = best["model"]

print(best_models)

{np.int64(0): 'RandomForest', np.int64(1): 'RandomForest'}


# SAVE BEST MODEL PER CLUSTER

In [23]:
for cluster_id in best_models:
    df_c = df[df["cluster"] == cluster_id]
    X = df_c[FEATURES]
    y = df_c[TARGET]

    model_name = best_models[cluster_id]

    if model_name == "LinearRegression":
        model = LinearRegression()
    elif model_name == "SGDRegressor":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("sgd", SGDRegressor(max_iter=2000, tol=1e-3))
        ])
    else:
        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            random_state=42,
            n_jobs=-1
        )

    model.fit(X, y)
    joblib.dump(model, f"{MODEL_OUTPUT_FOLDER}/model_cluster_{cluster_id}.pkl")

print("Models saved per cluster")


Models saved per cluster
